In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Load my heart disease dataset
df = pd.read_csv('heart_cleaned.csv')
print(f"Dataset shape: {df.shape}")

# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set: {X_train.shape}, Test set: {X_test.shape}")

Dataset shape: (1025, 14)

Training set: (820, 13), Test set: (205, 13)


In [18]:
# ============================================================================
# VARIETY 1: Only Encoded Dataset
# ============================================================================
print("\n" + "="*60)
print("VARIETY 1: Only Encoded Dataset")
print("="*60)

# Identify categorical columns
categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_columns = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical columns: {categorical_columns}")
print(f"Numerical columns: {numerical_columns}")

X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

# Encode categorical variables if any
if categorical_columns:
    for col in categorical_columns:
        le = LabelEncoder()
        X_train_raw[col] = le.fit_transform(X_train_raw[col])
        X_test_raw[col] = le.transform(X_test_raw[col])
    print(f"Encoded categorical columns: {categorical_columns}")

# Train logistic regression on raw data
lr_raw = LogisticRegression(random_state=42, max_iter=1000)
lr_raw.fit(X_train_raw, y_train)

# Predictions
y_pred_raw = lr_raw.predict(X_test_raw)

# Evaluation
accuracy_raw = accuracy_score(y_test, y_pred_raw)
f1_raw = f1_score(y_test, y_pred_raw)
precision_raw = precision_score(y_test, y_pred_raw)
recall_raw = recall_score(y_test, y_pred_raw)
cv_scores_raw = cross_val_score(lr_raw, X_train_raw, y_train, cv=5, scoring='accuracy')

print(f"Accuracy: {accuracy_raw:.4f}")
print(f"F1-Score: {f1_raw:.4f}")
print(f"Precision: {precision_raw:.4f}")
print(f"Recall: {recall_raw:.4f}")
print(f"Cross-validation scores: {cv_scores_raw}")
print(f"Mean CV accuracy: {cv_scores_raw.mean():.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_raw))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_raw))


VARIETY 1: Only Encoded Dataset
Categorical columns: ['sex', 'cp', 'fbs', 'exang', 'slope', 'thal']
Numerical columns: ['age', 'trestbps', 'chol', 'restecg', 'thalach', 'oldpeak', 'ca']
Encoded categorical columns: ['sex', 'cp', 'fbs', 'exang', 'slope', 'thal']
Accuracy: 0.8195
F1-Score: 0.8279
Precision: 0.8091
Recall: 0.8476
Cross-validation scores: [0.85365854 0.82317073 0.81707317 0.81707317 0.81097561]
Mean CV accuracy: 0.8244

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.79      0.81       100
           1       0.81      0.85      0.83       105

    accuracy                           0.82       205
   macro avg       0.82      0.82      0.82       205
weighted avg       0.82      0.82      0.82       205


Confusion Matrix:
[[79 21]
 [16 89]]


In [22]:
# ============================================================================
# VARIETY 2: Fully Preprocessed Dataset
# ============================================================================
print("\n" + "="*60)
print("VARIETY 2: Fully Preprocessed Dataset")
print("="*60)

# Use the already encoded data from Variety 1
X_train_processed = X_train_raw.copy()
X_test_processed = X_test_raw.copy()

# Step 2: Outlier Detection and Removal using Isolation Forest
print("Applying outlier detection...")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outlier_labels = iso_forest.fit_predict(X_train_processed)

# Remove outliers from training data
outlier_mask = outlier_labels == 1
X_train_no_outliers = X_train_processed[outlier_mask]
y_train_no_outliers = y_train[outlier_mask]

print(f"Training data before outlier removal: {X_train_processed.shape}")
print(f"Training data after outlier removal: {X_train_no_outliers.shape}")
print(f"Outliers removed: {sum(outlier_labels == -1)} samples")

# Step 3: Feature Selection
print("\nApplying feature selection...")
selector = SelectKBest(score_func=f_classif, k=8)  # Select top 8 features
X_train_selected = selector.fit_transform(X_train_no_outliers, y_train_no_outliers)
X_test_selected = selector.transform(X_test_processed)

# Get selected feature names and scores
selected_features_mask = selector.get_support()
selected_features = X_train_no_outliers.columns[selected_features_mask].tolist()
feature_scores = selector.scores_[selected_features_mask]

print(f"Selected {len(selected_features)} features:")
for feature, score in zip(selected_features, feature_scores):
    print(f"  {feature}: {score:.4f}")

# Step 4: Standardization
print("\nApplying standardization...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

print("Applied preprocessing: Encoding + Outlier Removal + Feature Selection + Standardization")

# Train logistic regression on preprocessed data
lr_processed = LogisticRegression(random_state=42, max_iter=1000)
lr_processed.fit(X_train_scaled, y_train_no_outliers)

# Predictions
y_pred_processed = lr_processed.predict(X_test_scaled)

# Evaluation
accuracy_processed = accuracy_score(y_test, y_pred_processed)
f1_processed = f1_score(y_test, y_pred_processed)
precision_processed = precision_score(y_test, y_pred_processed)
recall_processed = recall_score(y_test, y_pred_processed)
cv_scores_processed = cross_val_score(lr_processed, X_train_scaled, y_train_no_outliers, cv=5, scoring='accuracy')

print(f"Accuracy: {accuracy_processed:.4f}")
print(f"F1-Score: {f1_processed:.4f}")
print(f"Precision: {precision_processed:.4f}")
print(f"Recall: {recall_processed:.4f}")
print(f"Cross-validation scores: {cv_scores_processed}")
print(f"Mean CV accuracy: {cv_scores_processed.mean():.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_processed))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_processed))


VARIETY 2: Fully Preprocessed Dataset
Applying outlier detection...
Training data before outlier removal: (820, 13)
Training data after outlier removal: (779, 13)
Outliers removed: 41 samples

Applying feature selection...
Selected 8 features:
  sex: 69.3708
  cp: 162.3242
  thalach: 171.5710
  exang: 201.8695
  oldpeak: 184.4723
  slope: 103.5854
  ca: 114.4536
  thal: 228.4200

Applying standardization...
Applied preprocessing: Encoding + Outlier Removal + Feature Selection + Standardization
Accuracy: 0.8244
F1-Score: 0.8318
Precision: 0.8165
Recall: 0.8476
Cross-validation scores: [0.82051282 0.82692308 0.84615385 0.82692308 0.79354839]
Mean CV accuracy: 0.8228

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.80      0.82       100
           1       0.82      0.85      0.83       105

    accuracy                           0.82       205
   macro avg       0.82      0.82      0.82       205
weighted avg       0.82      0

In [26]:
# ============================================================================
# VARIETY 3: Fully Preprocessed + Hyperparameter Tuned Dataset
# ============================================================================
print("\n" + "="*60)
print("VARIETY 3: Fully Preprocessed + Hyperparameter Tuned Dataset")
print("="*60)

# Use the same preprocessed data from Variety 2
X_train_tune = X_train_scaled.copy()
X_test_tune = X_test_scaled.copy()
y_train_tune = y_train_no_outliers.copy()

# Use preprocessed data directly
X_train_final = X_train_tune
X_test_final = X_test_tune

# Manual Hyperparameter Tuning
print("\n--- Manual Hyperparameter Tuning Results ---")

# Try different parameter combinations
param_combinations = [
    {'C': 0.01, 'penalty': 'l2', 'solver': 'liblinear'},
    {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'},
    {'C': 1.0, 'penalty': 'l2', 'solver': 'liblinear'},
    {'C': 10.0, 'penalty': 'l2', 'solver': 'liblinear'},
    {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'},
    {'C': 1.0, 'penalty': 'l1', 'solver': 'liblinear'},
    {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'},
    {'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'}
]

best_accuracy = 0
best_params = None
best_model = None
tuning_results = []

for i, params in enumerate(param_combinations, 1):
    try:
        lr_tuned = LogisticRegression(**params, random_state=42, max_iter=1000)
        lr_tuned.fit(X_train_final, y_train_tune)

        # Cross-validation
        cv_scores = cross_val_score(lr_tuned, X_train_final, y_train_tune, cv=5, scoring='accuracy')
        mean_cv_accuracy = cv_scores.mean()

        tuning_results.append({
            'params': params,
            'cv_accuracy': mean_cv_accuracy,
            'cv_std': cv_scores.std()
        })

        print(f"Params {i}: C={params['C']}, penalty={params['penalty']}, solver={params['solver']}")
        print(f"  CV Accuracy: {mean_cv_accuracy:.4f}")

        if mean_cv_accuracy > best_accuracy:
            best_accuracy = mean_cv_accuracy
            best_params = params
            best_model = lr_tuned

    except Exception as e:
        print(f"Params {i} failed: {e}")

# Train final model with best parameters
print(f"\nBest parameters: {best_params}")
final_model = LogisticRegression(**best_params, random_state=42, max_iter=1000)
final_model.fit(X_train_final, y_train_tune)

# Predictions
y_pred_final = final_model.predict(X_test_final)

# Evaluation
accuracy_final = accuracy_score(y_test, y_pred_final)
f1_final = f1_score(y_test, y_pred_final)
precision_final = precision_score(y_test, y_pred_final)
recall_final = recall_score(y_test, y_pred_final)
cv_scores_final = cross_val_score(final_model, X_train_final, y_train_tune, cv=5, scoring='accuracy')

print(f"Final Model Accuracy: {accuracy_final:.4f}")
print(f"Final Model F1-Score: {f1_final:.4f}")
print(f"Final Model Precision: {precision_final:.4f}")
print(f"Final Model Recall: {recall_final:.4f}")
print(f"Final Model CV scores: {cv_scores_final}")
print(f"Mean CV accuracy: {cv_scores_final.mean():.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))


VARIETY 3: Fully Preprocessed + Hyperparameter Tuned Dataset

--- Manual Hyperparameter Tuning Results ---
Params 1: C=0.01, penalty=l2, solver=liblinear
  CV Accuracy: 0.8305
Params 2: C=0.1, penalty=l2, solver=liblinear
  CV Accuracy: 0.8228
Params 3: C=1.0, penalty=l2, solver=liblinear
  CV Accuracy: 0.8228
Params 4: C=10.0, penalty=l2, solver=liblinear
  CV Accuracy: 0.8228
Params 5: C=0.1, penalty=l1, solver=liblinear
  CV Accuracy: 0.8190
Params 6: C=1.0, penalty=l1, solver=liblinear
  CV Accuracy: 0.8215
Params 7: C=0.1, penalty=l2, solver=lbfgs
  CV Accuracy: 0.8228
Params 8: C=1.0, penalty=l2, solver=lbfgs
  CV Accuracy: 0.8228

Best parameters: {'C': 0.01, 'penalty': 'l2', 'solver': 'liblinear'}
Final Model Accuracy: 0.8537
Final Model F1-Score: 0.8611
Final Model Precision: 0.8378
Final Model Recall: 0.8857
Final Model CV scores: [0.83333333 0.83974359 0.83333333 0.83333333 0.81290323]
Mean CV accuracy: 0.8305

Classification Report:
              precision    recall  f1-sc

In [33]:
# ============================================================================
# COMPARISON OF ALL THREE VARIETIES
# ============================================================================
print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)

comparison_df = pd.DataFrame({
    'Model Variety': ['Only Encoded Dataset', 'Fully Preprocessed', 'Fully Preprocessed + Tuned'],
    'Test Accuracy': [accuracy_raw, accuracy_processed, accuracy_final],
    'F1-Score': [f1_raw, f1_processed, f1_final],
    'Precision': [precision_raw, precision_processed, precision_final],
    'Recall': [recall_raw, recall_processed, recall_final],
    'Mean CV Accuracy': [cv_scores_raw.mean(), cv_scores_processed.mean(), cv_scores_final.mean()],
    'CV Std': [cv_scores_raw.std(), cv_scores_processed.std(), cv_scores_final.std()],
    'Features Used': [X_train_raw.shape[1], X_train_scaled.shape[1], X_train_final.shape[1]]
})

print(comparison_df.round(4))

print(f"\nKey Observations:")
print(f"• Accuracy improvement from Only Encoded to Fully Preprocessed: {accuracy_processed - accuracy_raw:+.4f}")
print(f"• Accuracy improvement from Fully Preprocessed to Tuned: {accuracy_final - accuracy_processed:+.4f}")
print(f"• Total improvement: {accuracy_final - accuracy_raw:+.4f}")
print(f"• F1-Score improvement: {f1_final - f1_raw:+.4f}")

# Feature importance for interpretation
if hasattr(final_model, 'coef_') and best_params['penalty'] != 'l2':
    print(f"\nFeature Coefficients from final model:")
    for i, coef in enumerate(final_model.coef_[0]):
        print(f"  Component {i+1}: {coef:.4f}")
else:
    print("\nNote: L2 regularization doesn't provide feature coefficients for interpretation")

# Show top performing parameter combinations
print("\nTop 3 parameter combinations from tuning:")
tuning_df = pd.DataFrame(tuning_results).sort_values('cv_accuracy', ascending=False).head(3)
for idx, row in tuning_df.iterrows():
    params = row['params']
    print(f"  C={params['C']}, penalty={params['penalty']}, solver={params['solver']}: {row['cv_accuracy']:.4f}")

print("\nLimitations & Observations:")
print("• Logistic Regression assumes linear separability — may underperform on nonlinear datasets.")
print("• Future improvements: Try non-linear models (SVM, Random Forest) or deep learning methods. (which our other group members did)")



COMPARISON SUMMARY
                Model Variety  Test Accuracy  F1-Score  Precision  Recall  \
0        Only Encoded Dataset         0.8195    0.8279     0.8091  0.8476   
1          Fully Preprocessed         0.8244    0.8318     0.8165  0.8476   
2  Fully Preprocessed + Tuned         0.8537    0.8611     0.8378  0.8857   

   Mean CV Accuracy  CV Std  Features Used  
0            0.8244  0.0151             13  
1            0.8228  0.0170              8  
2            0.8305  0.0092              8  

Key Observations:
• Accuracy improvement from Only Encoded to Fully Preprocessed: +0.0049
• Accuracy improvement from Fully Preprocessed to Tuned: +0.0293
• Total improvement: +0.0341
• F1-Score improvement: +0.0332

Note: L2 regularization doesn't provide feature coefficients for interpretation

Top 3 parameter combinations from tuning:
  C=0.01, penalty=l2, solver=liblinear: 0.8305
  C=0.1, penalty=l2, solver=liblinear: 0.8228
  C=1.0, penalty=l2, solver=liblinear: 0.8228

Limitation